# TRIAGE on Kaggle T4×2 (free tier)

Full local pipeline, zero LLM API keys: generator `llama3.1:8b` + judge `qwen2.5:14b`, both via ollama.

**Before running: notebook Settings →**
- Accelerator: **GPU T4 x2**
- Internet: **ON** (requires phone-verified Kaggle account)

Session budget: 12 h wall clock, ~29 GB system RAM. `prepare --rebuild` is the long pole (2–5 h on 4 vCPU + T4 embedding). The benchmark checkpoints per question, so an interrupted run resumes safely.

**Judge options (edit `JUDGE_OLLAMA_MODEL` in cell 4):**
- `qwen2.5:14b` (default, recommended): 8b + 14b ≈ 13.7 GB — both fit one T4, no model swapping. Decent judging quality.
- `qwen2.5:7b`: faster pilot-grade fallback if VRAM gets tight.
- `qwen2.5:32b`: does NOT fit a 16 GB T4 (≈19 GB Q4); ollama CPU-offloads and judging crawls (~3–6 tok/s). Not recommended on this hardware — use the A100/Colab Pro or the `JUDGE_PROVIDER=huggingface` cloud hybrid for 32B+ quality.

In [ ]:
# 1. Clone the repo (idempotent: skips if present, pulls latest)
import sys
print(sys.version)
if sys.version_info >= (3, 13):
    print('Python 3.13 session detected - fine: cell 2 will build a Python 3.12')
    print('side-env automatically. All project cells run through {PY} - never plain python.')
!test -d TRIAGE || git clone --depth 1 https://github.com/Nirajd071/TRIAGE.git
%cd /kaggle/working/TRIAGE
!git pull --rebase 2>/dev/null; ls src/

In [ ]:
# 2. Install dependencies (10-20 min)
#    Python 3.13 session: builds a 3.12 side-env with micromamba.
#    All later cells run through {PY} - never plain python.
import sys
PY = sys.executable
if sys.version_info >= (3, 13):
    !curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /usr/local bin/micromamba
    !micromamba create -y -p /kaggle/tmp/py312 python=3.12
    PY = '/kaggle/tmp/py312/bin/python'
!{PY} -m pip install -q -r requirements.txt
#    Force numpy 2.x: the resolver may pick 1.26 and the newest pyarrow
#    (via datasets/chromadb) refuses to import on numpy<2.
!{PY} -m pip install -q "numpy>=2.0,<2.6"
#    scispacy NER models with --no-deps: metadata pins spacy<3.8,
#    but they run fine under spacy 3.8 (verified on the dev machine).
!{PY} -m pip install -q --no-deps https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz
!{PY} -m pip install -q --no-deps https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_jnlpba_md-0.5.4.tar.gz
print('python:', PY)
!{PY} -c "import chromadb, spacy, scispacy; print('stack OK:', chromadb.__version__, spacy.__version__, scispacy.__version__)"

In [ ]:
# 2b. GPU verification + fix. The EMBEDDING step of prepare needs the T4;
#    the graph build is CPU by design (networkx) - CPU there is normal.
#    PyPI's default torch is a CUDA-13 build that needs a very new driver;
#    if Kaggle's driver is older, torch silently falls back to CPU. This
#    cell detects that and reinstalls the CUDA-12.6 build (~2.5 GB).
!nvidia-smi | head -12
import subprocess
r = subprocess.run([PY, '-c', 'import torch; print(torch.cuda.is_available())'], capture_output=True, text=True)
cuda_ok = r.stdout.strip().endswith('True')
print('torch.cuda.is_available():', cuda_ok)
if not cuda_ok:
    print('CUDA unavailable - reinstalling torch with CUDA 12.6 runtime...')
    !{PY} -m pip install -q --force-reinstall torch --index-url https://download.pytorch.org/whl/cu126
    r2 = subprocess.run([PY, '-c', 'import torch; print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")'], capture_output=True, text=True)
    print('after fix:', r2.stdout.strip())
else:
    !{PY} -c "import torch; print('GPU:', torch.cuda.get_device_name(0))"

In [ ]:
# 3. ollama: install (needs zstd), start server on GPU 1, pull models (~14 GB)
#    T4 x2: pin ollama to GPU 1 so GPU 0 stays free for the embedding
#    step of prepare (they collided before -> CUDA OOM at Step 5).
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import os
os.environ['OLLAMA_KEEP_ALIVE'] = '-1'   # keep both models resident in VRAM
get_ipython().system_raw('CUDA_VISIBLE_DEVICES=1 ollama serve > /tmp/ollama.log 2>&1 &')
!sleep 5 && ollama pull llama3.1:8b && ollama pull qwen2.5:14b
!ollama list

In [ ]:
# 4. Environment file - set your WANDB_API_KEY if you want online telemetry,
#    else it stays local (results JSON + figures).
#    NOTE: WANDB_API_KEY is also the off-box backup channel for the
#    benchmark checkpoint (uploaded as a W&B artifact every 5 questions).
WANDB_API_KEY = ""   # optional

with open('.env', 'w') as f:
    f.write("LLM_PROVIDER=ollama\n")
    f.write("JUDGE_OLLAMA_MODEL=qwen2.5:14b\n")
    f.write("WANDB_MODE=disabled\n" if not WANDB_API_KEY else f"WANDB_API_KEY={WANDB_API_KEY}\n")
# Print keys only (redacted values) so the saved notebook log stays clean.
print('\n'.join(line.split('=')[0] + '=***' for line in open('.env').read().strip().split('\n') if line.strip()))


### Optional: restore artifacts from a previous session

Skip this on a first run. After saving artifacts (bottom of this notebook) as a private Kaggle dataset, add that dataset in the notebook's right panel ("+ Add Input"), then run the cell to skip the 2–5 h `prepare --rebuild`.

In [ ]:
# 4b. RESTORE (only if you added your 'triage-artifacts' dataset as Input)
RESTORE = False   # set True after your first successful prepare
if RESTORE:
    !mkdir -p data/processed data/vectordb
    !cp -r /kaggle/input/triage-artifacts/* data/processed/ 2>/dev/null || true
    !cp -r /kaggle/input/triage-artifacts/vectordb/* data/vectordb/ 2>/dev/null || true
    !ls -la data/processed/ data/vectordb/

In [ ]:
# 5. Quick sanity: unit tests (no GPU, ~5 s)
!{PY} -m pytest tests/test_api_server.py tests/test_judge_provider.py -q

In [ ]:
# 6. Build the knowledge graph + vector store (THE LONG CELL: 2-5 h)
#    Now also downloads PMC clinical articles in Step 1 (Europe PMC, keyless;
#    ~15 min for the default 1000 - change via PMC_NUM_ARTICLES in .env),
#    chunks them, then embeds ~794k PrimeKG triples on GPU 0.
#    Embed batch is capped at 128 by default so it fits alongside ollama.
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ.setdefault('PMC_NUM_ARTICLES', '1000')
!{PY} src/main.py prepare --rebuild

In [ ]:
# 6b. Snapshot the prepare artifacts NOW (not at the end of the notebook).
#     If the session later dies during the multi-hour benchmark, this file
#     is already in /kaggle/working, so the next session restores instead
#     of re-running the 2-5 h prepare. Re-run at the end is unnecessary:
#     the benchmark does not modify these files.
!tar czf /kaggle/working/triage_artifacts.tar.gz \
    data/processed/primekg_graph.gpickle data/processed/sapbert_node_index.faiss \
    data/processed/sapbert_node_names.json data/vectordb
!du -sh /kaggle/working/triage_artifacts.tar.gz


In [ ]:
# 7. Smoke test: one query through all three systems
!{PY} src/main.py demo

In [ ]:
# 8. Pilot benchmark (~2.5 h: 24 paper-protocol questions at ~6 min each).
#    Verify results before the full run.
!{PY} src/main.py base-paper --samples 24 --plot


In [ ]:
# 9. Full benchmark (SESSION 2): runs only when RUN_FULL_BENCHMARK=True.
#    Keep False for the first Save & Run All - prepare + pilot must finish
#    inside Kaggle's 9 h GPU commit cap, otherwise the commit is killed
#    mid-run, no version is saved, and the artifact tarball is lost.
#    Session 2: restore artifacts (cell 4b), set True, Save & Run All.
RUN_FULL_BENCHMARK = False
if RUN_FULL_BENCHMARK:
    !{PY} src/main.py base-paper --samples 50 --plot
else:
    print('Skipped full benchmark (RUN_FULL_BENCHMARK=False). Run it in session 2.')


In [ ]:
# 10. Inspect the results
import json
r = json.load(open('results/base_paper_benchmark_results.json'))
print(json.dumps({k: r[k] for k in ('scoring_based_assessment', 'retrieval_quality',
      'citation_precision', 'significance_testing')}, indent=2)[:2000])

In [ ]:
# 11. Optional: live API check (the FastAPI server from src/api/server.py)
get_ipython().system_raw(f'nohup {PY} src/main.py serve --host 127.0.0.1 --port 8000 > /tmp/serve.log 2>&1 &')
!sleep 8 && curl -s http://127.0.0.1:8000/api/triage/health
!curl -s -X POST http://127.0.0.1:8000/api/triage/query -H 'Content-Type: application/json' \
  -d '{"query":"Can Metformin be given to patients with severe renal failure given lactic acidosis risk?","config":{"tauEntropy":0.25}}' | head -c 600
!pkill -f "main.py serve"

In [ ]:
# 12. Copy the results next to the artifacts (the tarball itself was
#     already created in cell 6b right after prepare - nothing to redo).
!cp results/base_paper_checkpoint.jsonl results/base_paper_benchmark_results.json /kaggle/working/ 2>/dev/null
!ls -la /kaggle/working/
# NOTE: /kaggle/working output limit is ~20 GB; the tarball is a few GB.
#       /kaggle/working only survives if a notebook VERSION is saved -
#       headless Save & Run All does that automatically on completion.


## Troubleshooting

- **OOM in cell 6:** Kaggle gives ~29 GB RAM; the PrimeKG graph build is the peak. If it dies, there is no swap — restart and restore from the artifacts dataset instead of re-running prepare.
- **Judge model:** default `qwen2.5:14b` fits alongside the 8b generator on one T4. `qwen2.5:7b` if tight; `qwen2.5:32b` does not fit a T4 (heavy CPU offload, very slow) — for 32B+ quality use the `JUDGE_PROVIDER=huggingface` cloud hybrid instead.
- **ollama dead after restart:** re-run cell 3 (the server doesn't survive kernel restarts).
- **Checkpoint reuse:** re-running cell 9 with the same sample count skips already-evaluated questions and aggregates over the full set.
- **12 h limit:** cell 6 + cell 9 together can exceed it on a first run. Do prepare + pilot in session 1 (save artifacts, cell 12), full benchmark in session 2 (restore first, cell 4b).
- **Session killed mid-benchmark:** the checkpoint (`results/base_paper_checkpoint.jsonl`) and question cache are auto-uploaded to W&B as an artifact (`benchmark-checkpoint`, every 5 questions) as long as `WANDB_API_KEY` is set. Download the latest artifact version from the `hcad-rag` W&B project, drop the two files into `results/`, restore artifacts (cell 4b), and re-run cell 9 - it resumes exactly where it died.
